# Load Synthetic Network

Load generated networks from CSV (edge list + positions) and NetworKit binary files.

In [ ]:
import pandas as pd
import networkit as nk
import matplotlib.pyplot as plt

In [ ]:
# --- Set paths (edit these to point to your files) ---
DATA_DIR = r"..\data\output\SampleConfig_results_20260216_013024\A\20kX\synthetic"
BASE = "mosaic_100x100_synthetic_network_20260216_013646"

EDGELIST_CSV = f"{DATA_DIR}\\{BASE}_edgelist.csv"
POSITIONS_CSV = f"{DATA_DIR}\\{BASE}_positions.csv"
NKBIN_FILE = f"{DATA_DIR}\\{BASE}.nkbin"

## 1. Load from CSV

In [ ]:
edges = pd.read_csv(EDGELIST_CSV)
print(f"Edges: {len(edges):,}")
edges.head()

In [ ]:
positions = pd.read_csv(POSITIONS_CSV)
print(f"Nodes: {len(positions):,}")
positions.head()

In [ ]:
print(f"Edge weight stats:\n{edges['weight'].describe()}")

## 2. Load from NetworKit binary

In [ ]:
nk_graph = nk.readGraph(NKBIN_FILE, nk.Format.NetworkitBinary)
print(f"Nodes: {nk_graph.numberOfNodes():,}")
print(f"Edges: {nk_graph.numberOfEdges():,}")
print(f"Weighted: {nk_graph.isWeighted()}")

## 3. Quick analysis with NetworKit

In [ ]:
cc = nk.components.ConnectedComponents(nk_graph)
cc.run()
print(f"Connected components: {cc.numberOfComponents():,}")
sizes = cc.getComponentSizes()
print(f"Largest component: {max(sizes.values()):,} nodes")

In [ ]:
dd = sorted(nk.centrality.DegreeCentrality(nk_graph).run().scores(), reverse=True)
plt.figure(figsize=(8, 4))
plt.hist(dd, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Degree centrality")
plt.ylabel("Count")
plt.title("Degree centrality distribution")
plt.tight_layout()
plt.show()

## 4. Plot a subregion from CSV

In [ ]:
# Crop to a small region for visualisation
x_min, x_max = 0, 1000
y_min, y_max = 0, 1000

pos_crop = positions[(positions.x >= x_min) & (positions.x <= x_max) &
                     (positions.y >= y_min) & (positions.y <= y_max)]
edges_crop = edges[
    (edges.source_x >= x_min) & (edges.source_x <= x_max) &
    (edges.source_y >= y_min) & (edges.source_y <= y_max) &
    (edges.target_x >= x_min) & (edges.target_x <= x_max) &
    (edges.target_y >= y_min) & (edges.target_y <= y_max)
]

print(f"Subregion: {len(pos_crop):,} nodes, {len(edges_crop):,} edges")

fig, ax = plt.subplots(figsize=(8, 8))
for _, e in edges_crop.iterrows():
    ax.plot([e.source_x, e.target_x], [e.source_y, e.target_y],
            "b-", linewidth=0.3, alpha=0.5)
ax.scatter(pos_crop.x, pos_crop.y, s=0.5, c="red", zorder=2)
ax.set_aspect("equal")
ax.set_title(f"Subregion [{x_min}:{x_max}, {y_min}:{y_max}]")
plt.tight_layout()
plt.show()